In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_batch = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/seo_keyword_performance/seo_keyword_performance.csv")
)

print("Batch 02 rows:", df_batch.count())

StatementMeta(, 25976479-c6f3-4d91-9c08-480ab444cb69, 4, Finished, Available, Finished, False)

Batch 02 rows: 30000


In [3]:
gold_watermark_path = "abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Gold.Lakehouse/Tables/dbo/control_watermark"

df_watermark = (
    spark.read
    .format("delta")
    .load(gold_watermark_path)
)

keyword_watermark = (
    df_watermark
    .filter(F.col("table_name") == "Fact_Keyword_Performance")
    .select("last_watermark")
    .first()["last_watermark"]
)

print("Keyword watermark:", keyword_watermark)

StatementMeta(, 25976479-c6f3-4d91-9c08-480ab444cb69, 5, Finished, Available, Finished, False)

Keyword watermark: 2026-01-31 23:50:00


In [4]:
df_incremental = df_batch.filter(
    F.col("updated_timestamp") > F.lit(keyword_watermark)
)

StatementMeta(, 25976479-c6f3-4d91-9c08-480ab444cb69, 6, Finished, Available, Finished, False)

In [5]:
df_clean = (
    df_incremental
    .withColumn(
        "country_code",
        F.when(
            F.upper(F.trim(F.col("country_code"))).isin(
                "USA", "UNITED STATES", "US"
            ),
            F.lit("US")
        )
        .otherwise(F.upper(F.trim(F.col("country_code"))))
    )
    .withColumn(
        "device_type",
        F.initcap(F.trim(F.col("device_type")))
    )
)

StatementMeta(, 25976479-c6f3-4d91-9c08-480ab444cb69, 7, Finished, Available, Finished, False)

In [6]:
df_clean = df_clean.filter(
    (F.col("search_volume") >= 0) &
    (F.col("clicks") >= 0) &
    (F.col("impressions") >= 0) &
    (F.col("ranking_position") > 0)
)

StatementMeta(, 25976479-c6f3-4d91-9c08-480ab444cb69, 8, Finished, Available, Finished, False)

In [7]:
df_clean = df_clean.withColumn(
    "ctr",
    F.when(
        F.col("impressions") > 0,
        F.col("clicks") / F.col("impressions")
    ).otherwise(F.lit(0.0))
)

StatementMeta(, 25976479-c6f3-4d91-9c08-480ab444cb69, 9, Finished, Available, Finished, False)

In [8]:
df_clean = df_clean.withColumn(
    "ranking_bucket",
    F.when(F.col("ranking_position") <= 3, "Top 3")
    .when(F.col("ranking_position") <= 10, "Page 1")
    .when(F.col("ranking_position") <= 20, "11-20")
    .when(F.col("ranking_position") <= 50, "21-50")
    .when(F.col("ranking_position") <= 100, "51-100")
    .otherwise("100+")
)

StatementMeta(, 25976479-c6f3-4d91-9c08-480ab444cb69, 10, Finished, Available, Finished, False)

In [9]:
window_spec = Window.partitionBy(
    "keyword_performance_id"
).orderBy(
    F.col("updated_timestamp").desc()
)

df_clean = (
    df_clean
    .withColumn("_rn", F.row_number().over(window_spec))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

StatementMeta(, 25976479-c6f3-4d91-9c08-480ab444cb69, 11, Finished, Available, Finished, False)

In [10]:
df_clean.createOrReplaceTempView("keyword_incremental")

StatementMeta(, 25976479-c6f3-4d91-9c08-480ab444cb69, 12, Finished, Available, Finished, False)

In [11]:
spark.sql("""
MERGE INTO fact_keyword_performance AS target
USING keyword_incremental AS source
ON target.keyword_performance_id = source.keyword_performance_id

WHEN MATCHED
     AND source.updated_timestamp > target.updated_timestamp
THEN UPDATE SET *

WHEN NOT MATCHED
THEN INSERT *
""")

StatementMeta(, 25976479-c6f3-4d91-9c08-480ab444cb69, 13, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]